# ORIE 5355 Project 2026: optional fairness bonus

This bonus is due with the project report, not Part 1. Run `model_training/purchase_model_baseline.ipynb` first. This example loads its pickled model and writes prices for gamma = 0, 0.1, and 0.9 to a separate `agents/bonus_fairness_prices.csv` in your project repository. It does not change `agents/static_prices_submission.csv`. Replace this simple baseline with your own model and strategy. There are no capacity constraints or competitors.

Maximize average revenue per customer minus `gamma * abs(median0 - median1)`. The medians include all offered prices for the two values of `Covariate1` (feature 0), not only prices on sales.


In [ ]:
from pathlib import Path
import os
import sys

# Resolve paths once so every notebook uses the project folder.
project_folder = next((
    folder for parent in (Path.cwd(), *Path.cwd().parents)
    for folder in (parent, parent / 'project_code_2026_publicshare',
                   parent / 'project_2026/project_code_2026_publicshare',
                   parent / 'private_github/project_2026/project_code_2026_publicshare')
    if (folder / 'pricing_helpers.py').is_file()
), None)
if project_folder is None:
    raise FileNotFoundError('Open the project folder as your notebook workspace.')
os.chdir(project_folder)
sys.path.insert(0, str(project_folder))

import pickle
import numpy as np
import pandas as pd
from pricing_helpers import BONUS_COLUMNS, BONUS_PRICE_FILENAME, fairness_summary, evaluate_static_prices

with (project_folder / 'agents/helper_files_here/purchase_model.pkl').open('rb') as file:
    model = pickle.load(file)  # Only load pickle files you trust.
training = pd.read_csv(project_folder / 'data/train_prices_decisions_2026.csv')
validation_customers = pd.read_csv(project_folder / 'data/validation_user_info_2026.csv')
test_customers = pd.read_csv(project_folder / 'data/test_user_info_2026.csv')
covariates = ['Covariate1', 'Covariate2', 'Covariate3']

## A naive pricing strategy

For gamma 0, choose each customer's price independently to maximize price times predicted purchase probability on a small grid. For positive gamma, try five amounts of shrinkage toward the best common price. Weight 1 keeps personalized prices; weight 0 charges everyone the common price and has zero median-gap penalty. The price grid is a baseline choice, not a restriction on your solution.


In [ ]:
price_grid = np.linspace(0, 500, 101)  # Baseline grid, not a pricing restriction.

In [ ]:
def predicted_revenue(customers, prices):
    features = customers[covariates].assign(price_item=prices)
    # Clip regression predictions; for a classifier, you may need
    # predict_proba() instead, using the column for purchase = 1.
    probability = np.clip(model.predict(features), 0, 1)
    return np.asarray(prices) * probability


def choose_static_prices(customers, gamma):
    personalized_prices = np.zeros(len(customers))
    personalized_revenue = np.zeros(len(customers))
    common_revenues = []
    for price in price_grid:
        revenue = predicted_revenue(customers, price)
        better = revenue > personalized_revenue
        personalized_prices[better] = price
        personalized_revenue[better] = revenue[better]
        common_revenues.append(revenue.mean())
    common_price = price_grid[np.argmax(common_revenues)]

    best_score = -np.inf
    weights = [1] if gamma == 0 else np.linspace(0, 1, 5)
    for weight in weights:
        prices = weight * personalized_prices + (1 - weight) * common_price
        revenue = predicted_revenue(customers, prices)
        score = fairness_summary(prices, customers.Covariate1, gamma, revenue)['score']
        if score > best_score:
            best_score = score
            best_prices, best_revenue, best_weight = prices, revenue, weight
    return best_prices, best_revenue, best_weight


In [ ]:
def make_static_price_file(customers, filename, overwrite=True):
    filename = Path(filename)
    if filename.exists() and not overwrite:
        raise FileExistsError(f'{filename} already exists; explicitly allow replacement.')
    submission = customers[['user_index']].copy()
    predicted_results = []
    for gamma, (price_column, revenue_column) in BONUS_COLUMNS.items():
        prices, revenue, weight = choose_static_prices(customers, gamma)
        submission[price_column], submission[revenue_column] = prices, revenue
        predicted_results.append({'gamma': gamma, 'weight': weight, **fairness_summary(
            prices, customers.Covariate1, gamma, revenue)})
    submission.to_csv(filename, index=False)
    print('Wrote:', filename, '| Customers:', len(submission))
    return pd.DataFrame(predicted_results)


## Validation: calculate realized performance offline

Write prices for `validation_user_info_2026.csv`, then call the evaluator. It uses the supplied encrypted validation valuations internally and returns aggregate results, not the labels. It calculates realized average revenue, both group medians, `penalty = gamma * abs(median0 - median1)`, and `score = average_revenue - penalty`. Medians include **all** offered prices, even when no sale occurs. Compare predicted and realized revenue; repeated tuning can overfit this validation set.

The validation CSV is for local testing only, not the required submission. Rerunning this notebook replaces its generated CSVs by default. Set `overwrite_csvs = False` to protect files from replacement.


In [ ]:
overwrite_csvs = True
validation_file = project_folder / 'model_training/bonus_fairness_validation.csv'
print('Predicted validation performance:')
make_static_price_file(validation_customers, validation_file, overwrite=overwrite_csvs)


In [ ]:
print('Realized validation performance:')
evaluate_static_prices(validation_file, project_folder / 'data', bonus=True)


## Write the separate bonus CSV and submit in your project repo

Apply your strategy to `test_user_info_2026.csv` and write `agents/bonus_fairness_prices.csv` with seven columns: `user_index`, `price_item`, `expected_revenue`, `price_item_gamma_0.1`, `expected_revenue_gamma_0.1`, `price_item_gamma_0.9`, and `expected_revenue_gamma_0.9`. Include every test customer exactly once and fill all entries with finite, nonnegative numbers. Keep the required Part 1 CSV separate and unchanged. Validation customers have different IDs and are not the test submission. Test valuations are not supplied, so this table shows only **predicted** test performance.

Commit this completed notebook and the bonus CSV to your **project repository** by the report deadline, and link them in the optional bonus section of your report. Explain your approach and how it changed across gamma, compare revenue/medians/penalties/scores, and discuss what matching medians does and does not imply. What would you do differently if the entire offered-price distributions had to match? This last comparison is conceptual; no additional implementation is required.


In [ ]:
test_file = project_folder / 'agents' / BONUS_PRICE_FILENAME
print('Predicted test performance:')
make_static_price_file(test_customers, test_file, overwrite=overwrite_csvs)
